# Modern Object Detection II: Faster R-CNN

Two-stage / region-proposal detection.


## 0. Workshop introduction

Faster R-CNN does **not** classify every location in one shot.

It first asks: *where might an object be?* (Region Proposal Network)  
Then it asks: *what is in each of those regions, and how should the box be refined?* (second stage)


## 1. Learning objectives

- Contrast **one-stage** (YOLO) with **two-stage** (Faster R-CNN).
- Explain why an RPN exists.
- Relate anchors, proposals, RoI Align, classification, and NMS.
- Visually compare RPN proposals with final detections.


## How this workshop is structured

You will **not** implement neural-network layers from scratch.

The instructor cells already contain working functions for each important stage of the algorithm. Your job is to:

1. Read what each stage does and why it exists.
2. Assemble those stages in the correct order (a short coding task).
3. Change one or two parameters and watch the output change.

The demo cell is only a one-liner (`run_full_pipeline`) so you can see a result after Run all. **Do not copy that function for the assembly exercise** — wire the named stages listed in the student task.

Hands-on coding is intentionally light (~20–25% of the session). Most of the time is for understanding the pipeline.


## 2. Environment setup


In [ ]:
# TorchVision ships with Colab. We only need plotting extras if missing.\n!pip install -q matplotlib opencv-python-headless pillow


In [ ]:
import platform
import sys

print("Python version:", sys.version.split()[0])
print("Platform:", platform.platform())

import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory:", round(props.total_memory / 1024 ** 3, 2), "GB")
else:
    print("GPU: None")
    print("GPU memory: n/a")
    print("\nEnable a GPU: Runtime → Change runtime type → T4 GPU, then Restart session.")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)


## Troubleshooting

| Problem | Fix |
|---|---|
| CUDA is unavailable | `Runtime → Change runtime type → T4 GPU`, then restart and Run all |
| Package import fails | `Runtime → Restart session`, then Run all |
| Checkpoint download fails | Re-run the setup / model-load cell |
| Out of memory | Use the smaller default model, or a smaller image |
| A student cell has `???` | That is expected. Fill it in, or set `RUN_STUDENT_ASSEMBLY = False` to skip it |

Do not spend workshop time debugging package conflicts. Restart and Run all first.


## 3. Imports


In [ ]:
# ==========================================
# INSTRUCTOR PROVIDED — DO NOT MODIFY
# ==========================================

import urllib.error
import urllib.request
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
from matplotlib import patches
from PIL import Image

SAMPLE_IMAGES = {
    "bus": "https://raw.githubusercontent.com/ultralytics/ultralytics/main/ultralytics/assets/bus.jpg",
    "zidane": "https://raw.githubusercontent.com/ultralytics/ultralytics/main/ultralytics/assets/zidane.jpg",
    "cats": "http://images.cocodataset.org/val2017/000000039769.jpg",
    "living_room": "http://images.cocodataset.org/val2017/000000000139.jpg",
    "street": "http://images.cocodataset.org/val2017/000000037777.jpg",
}
SAMPLE_VIDEOS = {
    "demo": "https://github.com/ultralytics/assets/releases/download/v0.0.0/solutions_ci_demo.mp4",
}


def download_file(url, path="sample.bin"):
    path = Path(path)
    if path.exists():
        return path
    req = urllib.request.Request(url, headers={"User-Agent": "object-detection-workshop/1.0"})
    try:
        with urllib.request.urlopen(req) as resp:
            path.write_bytes(resp.read())
    except urllib.error.HTTPError as err:
        loc = err.headers.get("Location")
        if err.code in (301, 302, 303, 307, 308) and loc:
            return download_file(loc, path)
        raise
    return path


def download_image(url, path="sample.jpg"):
    return download_file(url, path)


def download_video(url, path="sample.mp4"):
    return download_file(url, path)


def load_image(path):
    """Load an RGB uint8 image as a NumPy array (H, W, 3)."""
    return np.array(Image.open(path).convert("RGB"))


def _class_color(cls_id):
    rng = np.random.RandomState(int(cls_id) * 17 + 3)
    return rng.randint(40, 230, size=3) / 255.0


def visualize_detections(
    image,
    boxes,
    scores=None,
    labels=None,
    names=None,
    title=None,
    max_dets=60,
    prompt=None,
):
    """Draw xyxy boxes. `names` maps class id → string."""
    fig, ax = plt.subplots(1, 1, figsize=(10, 7))
    ax.imshow(image)
    ax.axis("off")
    header = title or ""
    if prompt:
        header = (header + "  |  prompt: " + str(prompt)).strip(" |")
    if header:
        ax.set_title(header, fontsize=12)

    boxes = [] if boxes is None else list(boxes)[:max_dets]
    scores = [None] * len(boxes) if scores is None else list(scores)[:max_dets]
    labels = [None] * len(boxes) if labels is None else list(labels)[:max_dets]

    for box, score, label in zip(boxes, scores, labels):
        x1, y1, x2, y2 = [float(v) for v in box]
        cls_id = 0 if label is None else int(label)
        color = _class_color(cls_id)
        ax.add_patch(
            patches.Rectangle(
                (x1, y1),
                max(x2 - x1, 1.0),
                max(y2 - y1, 1.0),
                linewidth=2,
                edgecolor=color,
                facecolor="none",
            )
        )
        name = names.get(cls_id, str(cls_id)) if isinstance(names, dict) else (str(label) if label is not None else "")
        caption = name if score is None else f"{name} {float(score):.2f}"
        ax.text(
            x1,
            max(y1 - 4, 12),
            caption,
            color="white",
            fontsize=9,
            bbox=dict(facecolor=color, edgecolor="none", pad=2, alpha=0.85),
        )
    fig.tight_layout()
    plt.show()
    return fig


def annotate_frame(image, boxes, scores=None, labels=None, names=None, max_dets=60):
    """Draw xyxy boxes on an RGB frame; returns RGB uint8 array (no matplotlib)."""
    out = np.asarray(image).copy()
    boxes = [] if boxes is None else list(boxes)[:max_dets]
    scores = [None] * len(boxes) if scores is None else list(scores)[:max_dets]
    labels = [None] * len(boxes) if labels is None else list(labels)[:max_dets]

    for box, score, label in zip(boxes, scores, labels):
        x1, y1, x2, y2 = [int(round(float(v))) for v in box]
        cls_id = 0 if label is None else int(label)
        rgb = (_class_color(cls_id) * 255).astype(np.uint8)
        color = (int(rgb[2]), int(rgb[1]), int(rgb[0]))  # OpenCV uses BGR
        cv2.rectangle(out, (x1, y1), (x2, y2), color, 2)
        name = names.get(cls_id, str(cls_id)) if isinstance(names, dict) else (str(label) if label is not None else "")
        caption = name if score is None else f"{name} {float(score):.2f}"
        (tw, th), _ = cv2.getTextSize(caption, cv2.FONT_HERSHEY_SIMPLEX, 0.45, 1)
        cv2.rectangle(out, (x1, max(y1 - th - 6, 0)), (x1 + tw + 4, y1), color, -1)
        cv2.putText(out, caption, (x1 + 2, max(y1 - 4, th)), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 255, 255), 1, cv2.LINE_AA)
    return out


## 4. Load an example image


In [ ]:
# ==========================================
# INSTRUCTOR PROVIDED — DO NOT MODIFY
# ==========================================

IMAGE_PATH = download_image(SAMPLE_IMAGES["bus"], "bus.jpg")
image = load_image(IMAGE_PATH)
print("image:", image.shape)
plt.figure(figsize=(8, 6)); plt.imshow(image); plt.axis("off"); plt.title("Input image"); plt.show()


## 5–6. The Faster R-CNN pipeline

```text
Image
  → Backbone / FPN     feature maps
  → Anchors            default boxes on a grid
  → RPN                objectness + box deltas → region proposals
  → RoI Align          crop each proposal from the feature map
  → Classification + box regression
  → NMS
  → Final detections
```

```text
YOLO:          image → dense predictions everywhere → NMS
Faster R-CNN:  image → proposals → classify / refine those regions → NMS
```

The RPN exists so the expensive classifier only runs on a few hundred regions, not on every pixel.


In [ ]:
# ==========================================
# INSTRUCTOR PROVIDED — DO NOT MODIFY
# ==========================================

from torchvision.models.detection import FasterRCNN_ResNet50_FPN_Weights, fasterrcnn_resnet50_fpn
from torchvision.transforms.functional import to_tensor

weights = FasterRCNN_ResNet50_FPN_Weights.DEFAULT
frcnn = fasterrcnn_resnet50_fpn(weights=weights, box_score_thresh=0.05)
frcnn.to(DEVICE).eval()
FRCNN_NAMES = {i: n for i, n in enumerate(weights.meta["categories"])}
print("Loaded Faster R-CNN ResNet50-FPN | classes:", len(FRCNN_NAMES))


def to_image_tensor(image):
    """RGB uint8 HWC → float CHW in [0, 1], the format TorchVision expects."""
    return to_tensor(image).to(DEVICE)


@torch.no_grad()
def extract_features(image_tensor):
    """Backbone + FPN: a pyramid of feature maps."""
    orig = [tuple(image_tensor.shape[-2:])]
    images, _ = frcnn.transform([image_tensor])
    features = frcnn.backbone(images.tensors)
    return images, features, orig


@torch.no_grad()
def generate_anchors(images, features):
    """Anchors: default boxes placed on the feature maps (not yet object-aware)."""
    feats = list(features.values()) if isinstance(features, dict) else features
    return frcnn.rpn.anchor_generator(images, feats)


@torch.no_grad()
def generate_region_proposals(images, features):
    """Region Proposal Network: score anchors and keep the promising ones."""
    proposals, _ = frcnn.rpn(images, features)
    return proposals


def filter_proposals(proposals, top_n=100):
    """Keep the highest-scoring proposals for inspection."""
    return [p[:top_n] for p in proposals]


@torch.no_grad()
def roi_align(features, proposals, image_sizes):
    """Crop each proposal from the feature maps to a fixed-size grid."""
    return frcnn.roi_heads.box_roi_pool(features, proposals, image_sizes)


@torch.no_grad()
def classify_regions(features, proposals, image_sizes):
    """Second stage: classify each proposal and regress a tighter box (no NMS yet)."""
    box_features = roi_align(features, proposals, image_sizes)
    box_features = frcnn.roi_heads.box_head(box_features)
    class_logits, box_regression = frcnn.roi_heads.box_predictor(box_features)
    return class_logits, box_regression


@torch.no_grad()
def apply_nms_frcnn(class_logits, box_regression, proposals, image_sizes, orig_sizes, score_thresh=0.5, nms_thresh=0.5):
    """Convert logits to boxes, filter by score, then NMS. Maps boxes back to the original image."""
    old_score, old_nms = frcnn.roi_heads.score_thresh, frcnn.roi_heads.nms_thresh
    frcnn.roi_heads.score_thresh = score_thresh
    frcnn.roi_heads.nms_thresh = nms_thresh
    boxes, scores, labels = frcnn.roi_heads.postprocess_detections(
        class_logits, box_regression, proposals, image_sizes
    )
    frcnn.roi_heads.score_thresh, frcnn.roi_heads.nms_thresh = old_score, old_nms
    detections = [{"boxes": b, "scores": s, "labels": l} for b, s, l in zip(boxes, scores, labels)]
    detections = frcnn.transform.postprocess(detections, image_sizes, orig_sizes)
    det = detections[0]
    return {
        "boxes": det["boxes"].detach().cpu(),
        "scores": det["scores"].detach().cpu(),
        "labels": det["labels"].detach().cpu(),
        "names": FRCNN_NAMES,
    }


@torch.no_grad()
def proposals_in_original_image(proposals, images, orig_sizes):
    dummy = [{
        "boxes": proposals[0],
        "scores": torch.ones(len(proposals[0]), device=proposals[0].device),
        "labels": torch.zeros(len(proposals[0]), dtype=torch.int64, device=proposals[0].device),
    }]
    mapped = frcnn.transform.postprocess(dummy, images.image_sizes, orig_sizes)[0]
    return mapped["boxes"].detach().cpu()


def show_dets(image, dets, title=None, max_dets=80):
    visualize_detections(
        image,
        dets["boxes"].numpy(),
        dets["scores"].numpy() if "scores" in dets else None,
        dets["labels"].numpy() if "labels" in dets else None,
        names=dets.get("names"),
        title=title,
        max_dets=max_dets,
    )


@torch.no_grad()
def run_full_pipeline(image, score_thresh=0.5, nms_thresh=0.5, proposal_preview=80):
    """Black-box demo. For the assembly exercise, call the named stages yourself."""
    image_tensor = to_image_tensor(image)
    images, features, orig_sizes = extract_features(image_tensor)
    anchors = generate_anchors(images, features)
    proposals = generate_region_proposals(images, features)
    class_logits, box_regression = classify_regions(features, proposals, images.image_sizes)
    detections = apply_nms_frcnn(
        class_logits, box_regression, proposals, images.image_sizes, orig_sizes,
        score_thresh=score_thresh, nms_thresh=nms_thresh,
    )
    prop_boxes = proposals_in_original_image(filter_proposals(proposals, top_n=proposal_preview), images, orig_sizes)
    return {
        "images": images,
        "features": features,
        "orig_sizes": orig_sizes,
        "anchors": anchors,
        "proposals": proposals,
        "class_logits": class_logits,
        "box_regression": box_regression,
        "prop_boxes": prop_boxes,
        "detections": detections,
    }


print("Faster R-CNN helpers ready.")


## Instructor demo

Walk through the two-stage pipeline. This is the assembly students should write.


In [ ]:
# ==========================================
# INSTRUCTOR PROVIDED — DO NOT MODIFY
# ==========================================

image_tensor = to_image_tensor(image)
images, features, orig_sizes = extract_features(image_tensor)
anchors = generate_anchors(images, features)
proposals = generate_region_proposals(images, features)
print("FPN levels:", list(features.keys()) if isinstance(features, dict) else type(features))
print("anchors on first level:", 0 if not anchors else len(anchors[0]))
print("RPN proposals:", len(proposals[0]))

prop_vis = filter_proposals(proposals, top_n=80)
prop_boxes = proposals_in_original_image(prop_vis, images, orig_sizes)
show_dets(
    image,
    {"boxes": prop_boxes, "scores": torch.ones(len(prop_boxes)), "labels": torch.zeros(len(prop_boxes), dtype=torch.long), "names": {0: "proposal"}},
    title="RPN proposals (top 80)",
)

class_logits, box_regression = classify_regions(features, proposals, images.image_sizes)
detections = apply_nms_frcnn(class_logits, box_regression, proposals, images.image_sizes, orig_sizes, score_thresh=0.5, nms_thresh=0.5)
DEMO = {
    "images": images, "features": features, "orig_sizes": orig_sizes, "anchors": anchors,
    "proposals": proposals, "class_logits": class_logits, "box_regression": box_regression,
    "prop_boxes": prop_boxes, "detections": detections,
}
print("final detections:", len(detections["boxes"]))
show_dets(image, detections, title="Final detections after classification + NMS")


## 7. Student assembly

Assemble the two-stage pipeline. RoI Align is used *inside* `classify_regions`; you do not need to call it unless you want to inspect feature crops.


In [ ]:
# ==========================================
# STUDENT TASK
# ==========================================
RUN_STUDENT_ASSEMBLY = True  # solution notebook: assembly is filled in

if RUN_STUDENT_ASSEMBLY:
    image_tensor = to_image_tensor(image)
    images, features, orig_sizes = extract_features(image_tensor)
    proposals = generate_region_proposals(images, features)
    class_logits, box_regression = classify_regions(features, proposals, images.image_sizes)
    detections = apply_nms_frcnn(
        class_logits, box_regression, proposals, images.image_sizes, orig_sizes,
        score_thresh=0.5, nms_thresh=0.5,
    )
    show_dets(image, detections, title="Student assembly (solution)")
    print("final detections:", len(detections["boxes"]))
else:
    print('Skipping student assembly.')


## 8–9. Experiments


In [ ]:
# ==========================================
# STUDENT TASK
# ==========================================

SCORE = 0.5    # try 0.2 and 0.8
NMS = 0.5      # try 0.3 and 0.8
TOP_PROPOSALS = 80

prop_boxes = proposals_in_original_image(
    filter_proposals(DEMO["proposals"], top_n=TOP_PROPOSALS), DEMO["images"], DEMO["orig_sizes"]
)
show_dets(
    image,
    {"boxes": prop_boxes, "scores": torch.ones(len(prop_boxes)), "labels": torch.zeros(len(prop_boxes), dtype=torch.long), "names": {0: "proposal"}},
    title=f"RPN proposals (top {TOP_PROPOSALS})",
)

dets_exp = apply_nms_frcnn(
    DEMO["class_logits"], DEMO["box_regression"], DEMO["proposals"],
    DEMO["images"].image_sizes, DEMO["orig_sizes"],
    score_thresh=SCORE, nms_thresh=NMS,
)
show_dets(image, dets_exp, title=f"Final | score={SCORE}, nms={NMS}")
print("proposals shown:", len(prop_boxes), "| final detections:", len(dets_exp["boxes"]))


## 10. Think about it

1. Why does Faster R-CNN need an RPN if YOLO can skip that stage?
2. What is the difference between an **anchor**, a **proposal**, and a **final detection**?
3. Why are there usually far more proposals than final boxes?


## 11. Optional challenge

Call `roi_align` and print the tensor shape. You should get one fixed-size feature grid per proposal.


## 12. Summary

- Faster R-CNN is **two-stage**: propose, then classify/refine.
- YOLO predicts densely; Faster R-CNN spends compute on a shortlist of regions.

Next: **RT-DETR** — object queries and a Transformer decoder, without classical NMS.
